# Arquitectura OLAP en Spotify (Dataset Real 2024) 🎵
## Contexto:
Eres el Arquitecto de Datos Principal en una disquera. El equipo de ingeniería extrajo un dump transaccional masivo con las canciones más reproducidas de Spotify en 2024. El archivo plano es ancho, redundante y poco óptimo para análisis.

Tu misión es aplicar una tubería ETL para convertir este archivo crudo en un Esquema de Estrella en memoria, y posteriormente responder a 10 preguntas críticas del equipo de analítica utilizando el Cubo OLAP.

In [2]:
import pandas as pd
import numpy as np

# EXTRACT: Leemos directamente el archivo crudo desde el repositorio
url_dataset = "https://raw.githubusercontent.com/mricardo89/data-mining/main/Unit-III/datasets/Most%20Streamed%20Spotify%20Songs%202024.csv"

# Para evitar errores de codificación con caracteres especiales en nombres de artistas
spotify_raw = pd.read_csv(url_dataset, encoding='latin-1')

print("Muestra de datos crudos (OLTP):")
display(spotify_raw.head(3))
print("Columnas disponibles:", spotify_raw.columns.tolist())

Muestra de datos crudos (OLTP):


,Track,Album Name,Artist,Release Date,ISRC,All Time Rank,Track Score,Spotify Streams,Spotify Playlist Count,Spotify Playlist Reach,...,SiriusXM Spins,Deezer Playlist Count,Deezer Playlist Reach,Amazon Playlist Count,Pandora Streams,Pandora Track Stations,Soundcloud Streams,Shazam Counts,TIDAL Popularity,Explicit Track
0,MILLION DOLLAR BABY,Million Dollar Baby - Single,Tommy Richman,4/26/2024,QM24S2402528,1,725.4,"390,470,936","30,716","196,631,588",...,684,62.0,"17,598,718",114.0,"18,004,655","22,931","4,818,457","2,669,262",NaN,0
1,Not Like Us,Not Like Us,Kendrick Lamar,5/4/2024,USUG12400910,2,545.9,"323,703,884","28,113","174,597,137",...,3,67.0,"10,422,430",111.0,"7,780,028","28,444","6,623,075","1,118,279",NaN,1
2,i like the way you kiss me,I like the way you kiss me,Artemas,3/19/2024,QZJ842400387,3,538.4,"601,309,283","54,331","211,607,669",...,536,136.0,"36,321,847",172.0,"5,022,621","5,639","7,208,651","5,285,340",NaN,0


Columnas disponibles: ['Track', 'Album Name', 'Artist', 'Release Date', 'ISRC', 'All Time Rank', 'Track Score', 'Spotify Streams', 'Spotify Playlist Count', 'Spotify Playlist Reach', 'Spotify Popularity', 'YouTube Views', 'YouTube Likes', 'TikTok Posts', 'TikTok Likes', 'TikTok Views', 'YouTube Playlist Reach', 'Apple Music Playlist Count', 'AirPlay Spins', 'SiriusXM Spins', 'Deezer Playlist Count', 'Deezer Playlist Reach', 'Amazon Playlist Count', 'Pandora Streams', 'Pandora Track Stations', 'Soundcloud Streams', 'Shazam Counts', 'TIDAL Popularity', 'Explicit Track']


### Fase 2: Transform & Load (Construyendo el Almacén)

Observa las columnas del dataset. Tienes texto (Track, Artist, Date) mezclado con métricas puras (Spotify Streams, YouTube Views). 

**Reto Arquitectónico:**
1. Crea una **Dimensión Tiempo (`dim_tiempo`)**: Extrae las columnas relacionadas al año, mes y día de lanzamiento. Elimina duplicados y crea un `id_tiempo`.
2. Crea una **Dimensión Pista (`dim_pista`)**: Extrae el nombre del track, el artista y si es explícita. Crea un `id_pista`.
3. Crea la **Tabla de Hechos (`fact_streams`)**: Cruza los IDs y conserva **únicamente** métricas numéricas (ej. Spotify Streams, YouTube Views, Track Score) junto con las llaves foráneas. NO debe haber texto aquí.

In [3]:
# --- ESCRIBE TU PIPELINE ETL AQUÍ ---

# 1. Crear dim_tiempo
dim_tiempo = spotify_raw[['Release Date']].drop_duplicates().reset_index(drop=True)
dim_tiempo['id_tiempo'] = dim_tiempo.index + 1

fechas = pd.to_datetime(dim_tiempo['Release Date'])
dim_tiempo['Year'] = fechas.dt.year
dim_tiempo['Month'] = fechas.dt.month
dim_tiempo['Day'] = fechas.dt.day

dim_tiempo = dim_tiempo[['id_tiempo', 'Release Date', 'Year', 'Month', 'Day']]
dim_tiempo.head()

,id_tiempo,Release Date,Year,Month,Day
0,1,4/26/2024,2024,4,26
1,2,5/4/2024,2024,5,4
2,3,3/19/2024,2024,3,19
3,4,1/12/2023,2023,1,12
4,5,5/31/2024,2024,5,31


In [4]:
# 2. Crear dim_pista
dim_pista = spotify_raw[['Track', 'Artist', 'Explicit Track']].drop_duplicates().reset_index(drop=True)
dim_pista['id_pista'] = dim_pista.index + 1

dim_pista = dim_pista[['id_pista', 'Track', 'Artist', 'Explicit Track']]
dim_pista.head()


,id_pista,Track,Artist,Explicit Track
0,1,MILLION DOLLAR BABY,Tommy Richman,0
1,2,Not Like Us,Kendrick Lamar,1
2,3,i like the way you kiss me,Artemas,0
3,4,Flowers,Miley Cyrus,0
4,5,Houdini,Eminem,1


In [5]:
# 3. Crear fact_streams
df_fact = spotify_raw.merge(dim_pista, on=['Track', 'Artist', 'Explicit Track'], how='left')
df_fact = df_fact.merge(dim_tiempo[['Release Date', 'id_tiempo']], on='Release Date', how='left')

metric_cols = [
    'All Time Rank', 'Track Score', 'Spotify Streams', 'Spotify Playlist Count',
    'Spotify Playlist Reach', 'Spotify Popularity', 'YouTube Views', 'YouTube Likes',
    'TikTok Posts', 'TikTok Likes', 'TikTok Views', 'YouTube Playlist Reach',
    'Apple Music Playlist Count', 'AirPlay Spins', 'SiriusXM Spins',
    'Deezer Playlist Count', 'Deezer Playlist Reach', 'Amazon Playlist Count',
    'Pandora Streams', 'Pandora Track Stations', 'Soundcloud Streams',
    'Shazam Counts', 'TIDAL Popularity'
]

for col in metric_cols:
    df_fact[col] = pd.to_numeric(df_fact[col].astype(str).str.replace(',', ''), errors='coerce')

fact_streams = df_fact[['id_pista', 'id_tiempo'] + metric_cols].copy()
fact_streams.head()


,id_pista,id_tiempo,All Time Rank,Track Score,Spotify Streams,Spotify Playlist Count,Spotify Playlist Reach,Spotify Popularity,YouTube Views,YouTube Likes,...,AirPlay Spins,SiriusXM Spins,Deezer Playlist Count,Deezer Playlist Reach,Amazon Playlist Count,Pandora Streams,Pandora Track Stations,Soundcloud Streams,Shazam Counts,TIDAL Popularity
0,1,1,1,725.4,3.904709e+08,30716.0,196631588.0,92.0,8.427475e+07,1713126.0,...,40975.0,684.0,62.0,17598718.0,114.0,18004655.0,22931.0,4818457.0,2669262.0,NaN
1,2,2,2,545.9,3.237039e+08,28113.0,174597137.0,92.0,1.163470e+08,3486739.0,...,40778.0,3.0,67.0,10422430.0,111.0,7780028.0,28444.0,6623075.0,1118279.0,NaN
2,3,3,3,538.4,6.013093e+08,54331.0,211607669.0,92.0,1.225991e+08,2228730.0,...,74333.0,536.0,136.0,36321847.0,172.0,5022621.0,5639.0,7208651.0,5285340.0,NaN
3,4,4,4,444.9,2.031281e+09,269802.0,136569078.0,85.0,1.096101e+09,10629796.0,...,1474799.0,2182.0,264.0,24684248.0,210.0,190260277.0,203384.0,NaN,11822942.0,NaN
4,5,5,5,423.3,1.070349e+08,7223.0,151469874.0,88.0,7.737396e+07,3670188.0,...,12185.0,1.0,82.0,17660624.0,105.0,4493884.0,7006.0,207179.0,457017.0,NaN


In [6]:
# 4. (LOAD) Reconstruir el Cubo en memoria para las consultas
cubo_olap = fact_streams.merge(dim_pista, on='id_pista', how='inner')
cubo_olap = cubo_olap.merge(dim_tiempo, on='id_tiempo', how='inner')
display(cubo_olap.head(3))

,id_pista,id_tiempo,All Time Rank,Track Score,Spotify Streams,Spotify Playlist Count,Spotify Playlist Reach,Spotify Popularity,YouTube Views,YouTube Likes,...,Soundcloud Streams,Shazam Counts,TIDAL Popularity,Track,Artist,Explicit Track,Release Date,Year,Month,Day
0,1,1,1,725.4,390470936.0,30716.0,196631588.0,92.0,84274754.0,1713126.0,...,4818457.0,2669262.0,NaN,MILLION DOLLAR BABY,Tommy Richman,0,4/26/2024,2024,4,26
1,2,2,2,545.9,323703884.0,28113.0,174597137.0,92.0,116347040.0,3486739.0,...,6623075.0,1118279.0,NaN,Not Like Us,Kendrick Lamar,1,5/4/2024,2024,5,4
2,3,3,3,538.4,601309283.0,54331.0,211607669.0,92.0,122599116.0,2228730.0,...,7208651.0,5285340.0,NaN,i like the way you kiss me,Artemas,0,3/19/2024,2024,3,19


### Explica porqué solo hay 2 tablas dimensiones o si podrías incluir alguna otra. 

Solo hay 2 dimensiones porque el dataset original solo describe el contexto temporal y los atributos del track, mientras que casi todas las demas columnas son unas metricas cuantitativas que se van a la tabla de hechos, talvez se puede agregar una dimension de album, de artista y plataforma

### Fase 3: Consultas Analíticas (10 Retos)

El equipo de analítica te ha enviado 10 peticiones. Para cada una, debes:
1. Escribir el código en Pandas usando la matriz `cubo_olap` que construiste.
2. Identificar y escribir en un comentario qué tipo de operación OLAP estás aplicando: **Roll-up, Drill-down, Slice o Dice**.

In [8]:
# RETO 1: El CEO quiere ver el total acumulado de 'Spotify Streams' agrupado por 'Artist'.
# Operación OLAP: Roll-up (agregación subiendo en la jerarquía dimensional)
respuesta_1 = cubo_olap.groupby('Artist')['Spotify Streams'].sum()
display(respuesta_1)

# RETO 2: El equipo de marketing quiere ver todas las métricas únicamente de las canciones lanzadas en el año 2024.
# Operación OLAP: Slice (corte fijando un valor específico en la dimensión temporal)
respuesta_2 = cubo_olap[cubo_olap['Year'] == 2024]
display(respuesta_2.head(3))

# RETO 3: Finanzas requiere el total de 'Spotify Streams' desglosado primero por Año de Lanzamiento (Year) y luego detallado por Mes de Lanzamiento (Month).
# Operación OLAP: Drill-down (desglose jerárquico aumentando el nivel de detalle temporal)
respuesta_3 = cubo_olap.groupby(['Year', 'Month'])['Spotify Streams'].sum()
display(respuesta_3)

# RETO 4: Auditoría te pide extraer el rendimiento de las canciones que son explícitas Y que además fueron lanzadas en el año 2024.
# Operación OLAP: Dice (filtrado multidimensional sobre 2 dimensiones: contenido y tiempo)
respuesta_4 = cubo_olap[(cubo_olap['Explicit Track'] == 1) & (cubo_olap['Year'] == 2024)]
display(respuesta_4.head(3))

# RETO 5: Queremos disminuir el nivel de detalle y ver solamente el promedio del 'Track Score' agrupado por Año de Lanzamiento.
# Operación OLAP: Roll-up (resumen agregando por año y calculando promedio)
respuesta_5 = cubo_olap.groupby('Year')['Track Score'].mean()
display(respuesta_5)


Artist
"XY"                                           0.000000e+00
$OHO BANI                                      5.406556e+07
$uicideboy$                                    1.697447e+09
&ME                                            3.460163e+07
(G)I-DLE                                       8.769385e+08
                                                   ...     
ýýýýýýýýýýýýýýý ýýýýýýýýýýýýýýý                2.409532e+06
ýýýýýýýýýýýýýýý(ýýýýýýýýýýýýýýý)               5.484080e+05
ýýýýýýýýýýýýýýýýýý                             4.943057e+06
ýýýýýýýýýýýýýýýýýýýýý ýýýýýýýýýýýýýýýýýýýýý    4.560340e+05
ýýýýýýýýýýýýýýýýýýýýýýýý                       3.225791e+06
Name: Spotify Streams, Length: 1999, dtype: float64

,id_pista,id_tiempo,All Time Rank,Track Score,Spotify Streams,Spotify Playlist Count,Spotify Playlist Reach,Spotify Popularity,YouTube Views,YouTube Likes,...,Soundcloud Streams,Shazam Counts,TIDAL Popularity,Track,Artist,Explicit Track,Release Date,Year,Month,Day
0,1,1,1,725.4,390470936.0,30716.0,196631588.0,92.0,84274754.0,1713126.0,...,4818457.0,2669262.0,NaN,MILLION DOLLAR BABY,Tommy Richman,0,4/26/2024,2024,4,26
1,2,2,2,545.9,323703884.0,28113.0,174597137.0,92.0,116347040.0,3486739.0,...,6623075.0,1118279.0,NaN,Not Like Us,Kendrick Lamar,1,5/4/2024,2024,5,4
2,3,3,3,538.4,601309283.0,54331.0,211607669.0,92.0,122599116.0,2228730.0,...,7208651.0,5285340.0,NaN,i like the way you kiss me,Artemas,0,3/19/2024,2024,3,19


Year  Month
1987  7        1.879386e+09
1991  9        2.021910e+09
1994  1        1.810650e+09
1998  4        1.783399e+08
1999  6        1.405354e+09
                   ...     
2024  2        1.023077e+10
      3        1.263673e+10
      4        2.152035e+10
      5        1.005348e+10
      6        2.711085e+09
Name: Spotify Streams, Length: 191, dtype: float64

,id_pista,id_tiempo,All Time Rank,Track Score,Spotify Streams,Spotify Playlist Count,Spotify Playlist Reach,Spotify Popularity,YouTube Views,YouTube Likes,...,Soundcloud Streams,Shazam Counts,TIDAL Popularity,Track,Artist,Explicit Track,Release Date,Year,Month,Day
1,2,2,2,545.9,323703884.0,28113.0,174597137.0,92.0,116347040.0,3486739.0,...,6623075.0,1118279.0,NaN,Not Like Us,Kendrick Lamar,1,5/4/2024,2024,5,4
4,5,5,5,423.3,107034922.0,7223.0,151469874.0,88.0,77373957.0,3670188.0,...,207179.0,457017.0,NaN,Houdini,Eminem,1,5/31/2024,2024,5,31
7,8,8,8,375.8,675079153.0,40094.0,211236940.0,92.0,228382568.0,1439495.0,...,NaN,6063523.0,NaN,Gata Only,FloyyMenor,1,2/2/2024,2024,2,2


Year
1987    21.500000
1991    19.600000
1994    31.500000
1998    20.800000
1999    22.200000
2000    30.400000
2001    20.100000
2002    26.060000
2003    23.766667
2004    24.240000
2005    27.140000
2006    22.300000
2007    23.860000
2008    23.854545
2009    26.000000
2010    31.425926
2011    31.800000
2012    32.197872
2013    31.674545
2014    31.678571
2015    37.988542
2016    37.828455
2017    37.949765
2018    36.977510
2019    39.380405
2020    38.042222
2021    38.144118
2022    41.693228
2023    43.264594
2024    54.120058
Name: Track Score, dtype: float64

In [9]:
# RETO 6: Muestra el comportamiento analítico aislando únicamente los datos del artista "Bad Bunny".
# Operación OLAP: Slice (corte fijando la dimensión artista)
respuesta_6 = cubo_olap[cubo_olap['Artist'] == 'Bad Bunny']
display(respuesta_6.head(3))

# RETO 7: Aumenta el nivel de detalle: Muestra el total de 'YouTube Views' agrupado por Artista, y dentro de cada Artista, desglosado por Nombre de la Canción (Track).
# Operación OLAP: Drill-down (desglose jerárquico pasando de Artista a Canción individual)
respuesta_7 = cubo_olap.groupby(['Artist', 'Track'])['YouTube Views'].sum()
display(respuesta_7)

# RETO 8: Extrae un sub-cubo que contenga únicamente canciones lanzadas en el Mes 12 (Diciembre) del Año 2023.
# Operación OLAP: Dice (sub-cubo con filtro combinado de dos atributos temporales: Mes y Año)
respuesta_8 = cubo_olap[(cubo_olap['Month'] == 12) & (cubo_olap['Year'] == 2023)]
display(respuesta_8.head(3))

# RETO 9: Resume la información calculando el total histórico de 'Spotify Streams' de todos los tiempos, agrupado por Mes (para saber en qué mes históricamente se lanza la música más exitosa).
# Operación OLAP: Roll-up (agregación general acumulando métricas a nivel de Mes)
respuesta_9 = cubo_olap.groupby('Month')['Spotify Streams'].sum()
display(respuesta_9)

# RETO 10: Extrae un bloque de información filtrando las canciones de "Taylor Swift" que NO sean explícitas y hayan salido en 2024.
# Operación OLAP: Dice (sub-cubo filtrando a través de 3 dimensiones simultáneas: Artista, Contenido y Año)
respuesta_10 = cubo_olap[(cubo_olap['Artist'] == 'Taylor Swift') & (cubo_olap['Explicit Track'] == 0) & (cubo_olap['Year'] == 2024)]
display(respuesta_10)


,id_pista,id_tiempo,All Time Rank,Track Score,Spotify Streams,Spotify Playlist Count,Spotify Playlist Reach,Spotify Popularity,YouTube Views,YouTube Likes,...,Soundcloud Streams,Shazam Counts,TIDAL Popularity,Track,Artist,Explicit Track,Release Date,Year,Month,Day
63,61,57,64,184.8,1.811991e+09,173358.0,67603671.0,79.0,1.516863e+09,8870768.0,...,NaN,3393920.0,NaN,Me Porto Bonito,Bad Bunny,1,5/6/2022,2022,5,6
88,86,57,89,163.1,1.514240e+09,190692.0,77283361.0,77.0,1.366181e+09,9358441.0,...,NaN,5133847.0,NaN,Titï¿½ï¿½ Me Pregu,Bad Bunny,1,5/6/2022,2022,5,6
102,100,91,103,150.7,4.781509e+08,73306.0,51842133.0,78.0,1.133797e+08,1312763.0,...,4374049.0,2255465.0,NaN,MONACO,Bad Bunny,1,10/13/2023,2023,10,13


Artist                                       Track                                                              
"XY"                                         Come and Say - Original Mix                                            415159148.0
$OHO BANI                                    ZEIT, DASS SICH WAS DREHT                                                6744201.0
$uicideboy$                                  ...And to Those I Love, Thanks for Sticking Around                     269460083.0
                                             1000 Blunts                                                             87680985.0
                                             Are You Going to See the Rose in the Vase, or the Dust on the Table      2640169.0
                                                                                                                       ...     
ýýýýýýýýýýýýýýý ýýýýýýýýýýýýýýý              ýýýýýýýýýýýýýýýýýýýýýýýýýýýýýýýýýýýýýýýýýý                              44

,id_pista,id_tiempo,All Time Rank,Track Score,Spotify Streams,Spotify Playlist Count,Spotify Playlist Reach,Spotify Popularity,YouTube Views,YouTube Likes,...,Soundcloud Streams,Shazam Counts,TIDAL Popularity,Track,Artist,Explicit Track,Release Date,Year,Month,Day
61,59,55,62,186.5,1384.0,2.0,801.0,NaN,222076.0,5430.0,...,NaN,NaN,NaN,Very Cute Melody by Marimba Tone (39813),mitsu sound,0,12/31/2023,2023,12,31
94,92,84,95,155.9,649404552.0,27544.0,47789063.0,83.0,433839212.0,2750086.0,...,NaN,1380069.0,NaN,La Diabla,Xavi,0,12/2/2023,2023,12,2
97,95,87,98,153.1,359803580.0,26065.0,79152427.0,86.0,369321334.0,1839225.0,...,NaN,1002835.0,NaN,BELLAKEO,Peso Pluma,1,12/7/2023,2023,12,7


Month
1     3.652168e+11
2     1.333565e+11
3     1.638041e+11
4     1.462846e+11
5     1.779293e+11
6     1.484293e+11
7     1.503832e+11
8     1.426826e+11
9     1.363626e+11
10    1.672667e+11
11    1.734614e+11
12    1.022498e+11
Name: Spotify Streams, dtype: float64

,id_pista,id_tiempo,All Time Rank,Track Score,Spotify Streams,Spotify Playlist Count,Spotify Playlist Reach,Spotify Popularity,YouTube Views,YouTube Likes,...,Soundcloud Streams,Shazam Counts,TIDAL Popularity,Track,Artist,Explicit Track,Release Date,Year,Month,Day
16,17,17,17,297.6,395433400.0,12784.0,177932568.0,89.0,104673836.0,2484083.0,...,377734.0,1210029.0,NaN,Fortnight (feat. Post Malone),Taylor Swift,0,4/18/2024,2024,4,18
866,861,17,862,52.3,166168092.0,5004.0,25607992.0,78.0,16423009.0,244859.0,...,NaN,116673.0,NaN,Whoï¿½ï¿½ï¿½s Afraid of Little O,Taylor Swift,0,4/18/2024,2024,4,18
1013,1007,17,1008,47.8,175613480.0,4454.0,20883678.0,84.0,15237337.0,222750.0,...,NaN,67886.0,NaN,"So Long, London",Taylor Swift,0,4/18/2024,2024,4,18
1167,1159,17,1161,44.0,161916445.0,3784.0,17036800.0,83.0,13509432.0,180327.0,...,NaN,47529.0,NaN,My Boy Only Breaks His Favorite Toys,Taylor Swift,0,4/18/2024,2024,4,18
1270,1260,17,1264,42.1,165611453.0,4418.0,21928475.0,78.0,9873283.0,96083.0,...,NaN,79964.0,NaN,Guilty as Sin?,Taylor Swift,0,4/18/2024,2024,4,18
2484,2454,183,2478,28.6,97601615.0,3397.0,13075013.0,87.0,5528058.0,78517.0,...,NaN,NaN,NaN,So High School,Taylor Swift,0,4/19/2024,2024,4,19
2517,2486,17,2513,28.3,106201248.0,2992.0,15108991.0,NaN,5521347.0,70034.0,...,NaN,24502.0,NaN,The Alchemy,Taylor Swift,0,4/18/2024,2024,4,18
2528,2497,17,2525,28.2,250430303.0,648.0,818594.0,NaN,NaN,NaN,...,NaN,NaN,NaN,I Can Do It With a Broken Heart,Taylor Swift,0,4/18/2024,2024,4,18
2776,2744,183,2764,26.7,88244010.0,2659.0,8584946.0,NaN,7381186.0,96742.0,...,NaN,32559.0,NaN,imgonnagetyouback,Taylor Swift,0,4/19/2024,2024,4,19
2816,2784,17,2811,26.5,105424914.0,2473.0,7785632.0,NaN,7227379.0,113318.0,...,NaN,21372.0,NaN,Fresh Out The Slammer,Taylor Swift,0,4/18/2024,2024,4,18
